# ETAPA 10 - Avaliação comparativa do Qwen3-8B
Compara o baseline oficial já salvo com **Qwen3-8B + adapter QLoRA** e **Qwen3-8B + adapter QLoRA + RAG**. Use Google Colab com **Tesla T4 ou GPU compatível com pelo menos 14 GB de VRAM** e runtime **Python 3.12**.

In [ ]:
!nvidia-smi

In [ ]:
import platform
print(platform.python_version())
assert platform.python_version_tuple()[:2] == ('3', '12'), 'Selecione um runtime com Python 3.12'

In [ ]:
import os, subprocess
REPOSITORY = 'https://github.com/mo1sess/tech-challenge-fase3.git'
PROJECT = '/content/tech-challenge-fase3'
if not os.path.exists(PROJECT):
    subprocess.run(['git', 'clone', REPOSITORY, PROJECT], check=True)
else:
    subprocess.run(['git', '-C', PROJECT, 'pull', '--ff-only'], check=True)
os.chdir(PROJECT)
print(os.getcwd())

In [ ]:
%pip install -q -e . -r requirements/evaluation-gpu.txt

In [ ]:
import torch, transformers, peft, accelerate, bitsandbytes, chromadb, sentence_transformers
versions = {
    'torch': torch.__version__, 'transformers': transformers.__version__,
    'peft': peft.__version__, 'accelerate': accelerate.__version__,
    'bitsandbytes': bitsandbytes.__version__, 'chromadb': chromadb.__version__,
    'sentence_transformers': sentence_transformers.__version__,
    'cuda': torch.cuda.is_available(),
    'gpu': torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
}
print(versions)
assert torch.cuda.is_available(), 'A avaliação oficial exige GPU CUDA'

In [ ]:
!python scripts/validate_model_evaluation.py

In [ ]:
# Reconstrói os 5 protocolos sintéticos. Os embeddings permanecem em CPU.
!python scripts/build_rag_index.py

## Envie o adapter real da ETAPA 5
Na próxima célula, selecione `qwen3_8b_qlora_stage5_evidence.zip`. O hash do `adapter_model.safetensors` será comparado ao manifesto oficial antes da inferência.

In [ ]:
from pathlib import Path
from google.colab import files
from clinical_assistant.acquisition.common import safe_extract_zip
uploaded = files.upload()
archives = [Path(name) for name in uploaded if name.lower().endswith('.zip')]
assert len(archives) == 1, 'Envie exatamente um arquivo ZIP da ETAPA 5'
evidence_root = Path('/content/stage5_evidence')
safe_extract_zip(archives[0], evidence_root)
adapters = [p.parent for p in evidence_root.rglob('adapter_model.safetensors') if p.parent.name == 'adapter']
assert len(adapters) == 1, f'Esperado um adapter final; encontrados: {adapters}'
ADAPTER = adapters[0]
print('Adapter:', ADAPTER)

## Smoke test
Executa 2 casos por variante. O resultado é marcado como `smoke_test` e não pode ser usado como comparação oficial.

In [ ]:
subprocess.run(['python', 'scripts/run_model_evaluation.py', str(ADAPTER), '--limit', '2'], check=True)

## Execução oficial completa
Executa os 24 casos nas duas variantes ajustadas (48 gerações). Não use `--limit`.

In [ ]:
subprocess.run(['python', 'scripts/run_model_evaluation.py', str(ADAPTER)], check=True)

In [ ]:
import json
runs = sorted(Path('outputs/evaluation').glob('evaluation-*'))
official = [run for run in runs if json.loads((run / 'summary.json').read_text())['complete']]
assert official, 'Nenhuma execução oficial completa foi encontrada'
RUN = official[-1]
subprocess.run(['python', 'scripts/validate_model_evaluation.py', '--run', str(RUN)], check=True)
summary = json.loads((RUN / 'summary.json').read_text(encoding='utf-8'))
print(json.dumps({
    'run_directory': str(RUN),
    'complete': summary['complete'],
    'cases_per_variant': summary['cases_per_variant'],
    'comparisons': summary['comparisons'],
    'peak_gpu_memory_gb': summary['peak_gpu_memory_gb'],
}, ensure_ascii=False, indent=2))

In [ ]:
from IPython.display import Markdown, display
display(Markdown((RUN / 'comparison.md').read_text(encoding='utf-8')))

In [ ]:
import shutil
archive = shutil.make_archive('/content/qwen3_8b_stage10_evaluation_evidence', 'zip', RUN)
print(archive)
files.download(archive)